# Optimized Thai Election OCR Pipeline
## Key Optimizations:
1. **Preprocessing**: Resize 1536px + CLAHE contrast + JPEG 85 → 70% fewer tokens
2. **Prompt**: ส่งชื่อพรรคที่คาดหวังใน prompt → model แค่จับคู่ ไม่ต้อง free-read
3. **Schema**: enum constraint บังคับให้ตอบเฉพาะชื่อพรรคที่ถูกต้อง
4. **Post-process**: fuzzy match + vote range validation + digit count check

In [ ]:
# === Cell 1: Install ===
!pip install -q openai google-generativeai rapidfuzz tqdm Pillow pandas kaggle opencv-python-headless 2>/dev/null
print('Done.')

In [ ]:
# === Cell 2: Imports & Config ===
import os, re, json, time, io, cv2, base64
import numpy as np
from pathlib import Path
from collections import defaultdict

import pandas as pd
from PIL import Image, ImageEnhance
from tqdm.notebook import tqdm
from openai import OpenAI
from rapidfuzz import fuzz
from google.colab import userdata

# === OpenRouter (PAID - no RPD limit!) ===
OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
MODEL = 'google/gemini-3-flash-preview'  # 98% accuracy on test!

client = OpenAI(
    base_url='https://openrouter.ai/api/v1',
    api_key=OPENROUTER_API_KEY,
)

DATA_DIR = Path('/content/data')
IMAGE_DIR = DATA_DIR / 'images'
SAMPLE_LABELS_DIR = DATA_DIR / 'sample_labels'
SUBMISSION_TEMPLATE = DATA_DIR / 'submission_template.csv'
OUTPUT_DIR = Path('/content/output')
OUTPUT_DIR.mkdir(exist_ok=True)
CHECKPOINT_FILE = OUTPUT_DIR / 'ocr_results.json'

THAI_TO_ARABIC = str.maketrans('\u0e50\u0e51\u0e52\u0e53\u0e54\u0e55\u0e56\u0e57\u0e58\u0e59', '0123456789')

print(f'Model: {MODEL} via OpenRouter (PAID)')
print(f'Image: 2048px')

In [ ]:
# === Cell 3: Download Data ===
os.makedirs('/root/.kaggle', exist_ok=True)
username = userdata.get('KAGGLE_USERNAME')
key = userdata.get('KAGGLE_KEY')
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({'username': username, 'key': key}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)

COMPETITION = 'super-ai-engineer-season-6-ocr-2569'
!kaggle competitions download -c {COMPETITION} -p /content/

import glob as glob_module
for zf in glob_module.glob('/content/*.zip'):
    !unzip -qo "{zf}" -d /content/data

# Auto-detect correct path (handle nested data/data/)
if (DATA_DIR / 'data' / 'images').exists():
    IMAGE_DIR = DATA_DIR / 'data' / 'images'
    SAMPLE_LABELS_DIR = DATA_DIR / 'data' / 'sample_labels'
    SUBMISSION_TEMPLATE = DATA_DIR / 'data' / 'submission_template.csv'
    print('Detected nested path: /content/data/data/')
elif (DATA_DIR / 'images').exists():
    pass  # Already correct
else:
    # Try to find images anywhere under /content
    import subprocess
    result = subprocess.run(['find', '/content/data', '-name', '*.png', '-type', 'f'], capture_output=True, text=True)
    if result.stdout.strip():
        first = Path(result.stdout.strip().split('\n')[0])
        IMAGE_DIR = first.parent
        SAMPLE_LABELS_DIR = IMAGE_DIR.parent / 'sample_labels'
        SUBMISSION_TEMPLATE = IMAGE_DIR.parent / 'submission_template.csv'
        print(f'Found images at: {IMAGE_DIR}')

imgs = list(IMAGE_DIR.glob('*.png')) if IMAGE_DIR.exists() else []
print(f'Images: {len(imgs)} | Template: {SUBMISSION_TEMPLATE.exists()} | Labels: {SAMPLE_LABELS_DIR.exists()}')

In [ ]:
# === Cell 4: Group Documents & Build Expected Parties ===
template_df = pd.read_csv(SUBMISSION_TEMPLATE)
print(f'Template: {template_df.shape[0]} rows, {template_df.doc_id.nunique()} docs')

def group_documents(image_dir):
    docs = defaultdict(list)
    pattern = re.compile(r'^(.+?)(?:_page(\d+))?\.png$')
    for p in sorted(image_dir.glob('*.png')):
        m = pattern.match(p.name)
        if m:
            doc_id = m.group(1)
            page = int(m.group(2)) if m.group(2) else 1
            docs[doc_id].append((page, str(p)))
    for d in docs:
        docs[d].sort()
    return dict(docs)

documents = group_documents(IMAGE_DIR)
doc_types = {d: ('constituency' if d.startswith('constituency') else 'partylist') for d in documents}

# Expected parties per doc (KEY: this is used for enum constraint)
doc_expected = defaultdict(list)
for _, row in template_df.iterrows():
    doc_expected[row['doc_id']].append({
        'id': row['id'],
        'party_name': row['party_name'],
        'row_num': row['row_num'],
    })

template_doc_rows = {d: len(rows) for d, rows in doc_expected.items()}
print(f'Documents: {len(documents)} | Constituency: {sum(1 for t in doc_types.values() if t=="constituency")} | Party list: {sum(1 for t in doc_types.values() if t=="partylist")}')

In [ ]:
# === Cell 5: Image Preprocessing ===

def preprocess_image(img_path, target_width=2048):
    """Optimized preprocessing: 2048px = sweet spot for accuracy."""
    img_cv = cv2.imread(img_path)
    if img_cv is None:
        return Image.open(img_path).convert('RGB')
    
    h, w = img_cv.shape[:2]
    if w > target_width:
        ratio = target_width / w
        img_cv = cv2.resize(img_cv, (target_width, int(h * ratio)), interpolation=cv2.INTER_LANCZOS4)
    
    # CLAHE
    lab = cv2.cvtColor(img_cv, cv2.COLOR_BGR2LAB)
    clahe = cv2.createCLAHE(clipLimit=1.5, tileGridSize=(8, 8))
    lab[:, :, 0] = clahe.apply(lab[:, :, 0])
    img_cv = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
    
    img_pil = Image.fromarray(cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB))
    img_pil = ImageEnhance.Sharpness(img_pil).enhance(1.3)
    return img_pil

def image_to_base64(img_pil):
    buf = io.BytesIO()
    img_pil.save(buf, format='JPEG', quality=90)
    return base64.b64encode(buf.getvalue()).decode('utf-8')

test_img = sorted(IMAGE_DIR.glob('*.png'))[0]
orig = Image.open(test_img)
processed = preprocess_image(str(test_img))
print(f'Original: {orig.size[0]}x{orig.size[1]} | Processed: {processed.size[0]}x{processed.size[1]}')

In [ ]:
# === Cell 6: Extraction Engine (OpenRouter) ===

def clean_votes(raw):
    if not raw or str(raw) in ('None', 'null', ''):
        return '0'
    s = str(raw).translate(THAI_TO_ARABIC)
    s = re.sub(r'[^0-9]', '', s)
    return s if s else '0'


def repair_json_text(text):
    """Aggressively repair malformed JSON from LLM output."""
    text = text.strip()
    if '```' in text:
        text = re.sub(r'^```(?:json)?\s*\n?', '', text)
        text = re.sub(r'\n?\s*```\s*$', '', text)
        text = text.strip()
    
    idx = text.find('[')
    if idx == -1:
        return []
    text = text[idx:]
    
    # Fix unquoted number values with commas: "votes": 1,591 → "votes": "1591"
    text = re.sub(
        r'"votes"\s*:\s*([๐-๙\d][\d๐-๙,\.]*)',
        lambda m: f'"votes": "{m.group(1).replace(",", "")}"',
        text
    )
    text = re.sub(r',\s*]', ']', text)
    text = re.sub(r',\s*}', '}', text)
    
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    
    # Close truncated array
    last_brace = text.rfind('}')
    if last_brace > 0:
        truncated = re.sub(r',\s*$', '', text[:last_brace+1]) + ']'
        try:
            return json.loads(truncated)
        except json.JSONDecodeError:
            pass
    
    # Last resort: regex extract
    objects = []
    for m in re.finditer(r'"party_name"\s*:\s*"([^"]+)"[^}]*"votes"\s*:\s*"?([^"}\s,]+)"?', text):
        objects.append({'party_name': m.group(1), 'votes': m.group(2).replace(',', '')})
    return objects


def build_prompt(doc_type, expected_parties):
    party_list_str = '\n'.join(f'- {p}' for p in expected_parties)
    
    if doc_type == 'constituency':
        table_hint = """โครงสร้างตาราง (แบบแบ่งเขต): 4 คอลัมน์
| หมายเลขประจำตัว | ชื่อ-สกุลผู้สมัคร | สังกัดพรรคการเมือง | ได้คะแนน |

คอลัมน์ "ได้คะแนน" (ขวาสุด) มีรูปแบบ:
  ตัวเลข (อารบิก 0-9 หรือ เลขไทย ๐-๙) อยู่ด้านหน้า ตามด้วยตัวหนังสือไทยสะกดจำนวนในวงเล็บ
  ตัวอย่าง: 34,167 (สามหมื่นสี่พันหนึ่งร้อยหกสิบเจ็ด)
  ตัวอย่าง: ๓๔,๑๖๗ (สามหมื่นสี่พันหนึ่งร้อยหกสิบเจ็ด)

ต้องอ่านชื่อ "พรรคการเมือง" จากคอลัมน์ที่ 3 (สังกัดพรรค) ไม่ใช่ชื่อผู้สมัคร"""
    else:
        table_hint = """โครงสร้างตาราง (แบบบัญชีรายชื่อ): 3 คอลัมน์
| หมายเลขของบัญชีรายชื่อ | ชื่อพรรคการเมือง | ได้คะแนน |

คอลัมน์ "ได้คะแนน" (ขวาสุด) มีรูปแบบ:
  ตัวเลข (อารบิก 0-9 หรือ เลขไทย ๐-๙) อยู่ด้านหน้า ตามด้วยตัวหนังสือไทยสะกดจำนวนในวงเล็บ
  ตัวอย่าง: 1,819 (หนึ่งพันแปดร้อยสิบเก้า)
  ตัวอย่าง: 39 (สามสิบเก้า)"""

    return f"""คุณคือระบบ OCR สำหรับเอกสารผลเลือกตั้ง สส. ของไทย (แบบ สส.6/1)
ภาพเหล่านี้คือหน้าต่างๆ จากเอกสารเดียวกัน ให้อ่านตารางคะแนนทุกหน้า

{table_hint}

รายชื่อพรรคที่ต้องดึงคะแนน ({len(expected_parties)} พรรค):
{party_list_str}

วิธีอ่านคะแนน (สำคัญมาก):
1. อ่านตัวเลขที่อยู่ด้านหน้าวงเล็บในคอลัมน์ "ได้คะแนน" เป็นหลัก
2. ใช้ตัวหนังสือไทยในวงเล็บ เพื่อ cross-check ว่าตัวเลขถูกต้อง
3. ถ้าเป็นเลขไทย (๐๑๒๓๔๕๖๗๘๙) ให้แปลงเป็นเลขอารบิก (0123456789)
4. ลบ comma ออกจากตัวเลข เช่น 14,813 → 14813
5. votes ต้องเป็น string เสมอ เช่น "14813"

กฎ:
- ใช้ชื่อพรรคตามรายชื่อด้านบนเท่านั้น ห้ามเปลี่ยนชื่อ
- ห้ามอ่านหมายเลขประจำตัว/หมายเลขพรรค มาเป็นคะแนน
- ต้องครบทุกพรรค รวมจากทุกหน้า ห้ามซ้ำ
- หน้าแรกเป็นหน้าสรุป ไม่มีตาราง ให้ข้ามไป

ตอบ JSON array เท่านั้น:
[{{"party_name": "ชื่อพรรค", "votes": "12345"}}]"""


def extract_document(doc_id, page_paths, doc_type, expected_parties):
    """Send preprocessed images to OpenRouter."""
    prompt = build_prompt(doc_type, expected_parties)
    
    content = []
    for _, page_path in page_paths:
        img = preprocess_image(page_path)
        b64 = image_to_base64(img)
        content.append({
            'type': 'image_url',
            'image_url': {'url': f'data:image/jpeg;base64,{b64}'}
        })
    content.append({'type': 'text', 'text': prompt})
    
    last_error = None
    for attempt in range(3):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[{'role': 'user', 'content': content}],
                temperature=0.0,
                max_tokens=8192,
            )
            
            text = response.choices[0].message.content.strip()
            rows = repair_json_text(text)
            if rows:
                return rows
            
            last_error = f'JSON parse failed (len={len(text)})'
            if attempt < 2:
                print(f'    JSON retry {attempt+1}...')
                time.sleep(3)
                continue
        
        except Exception as e:
            last_error = str(e)
            if '429' in last_error or 'rate' in last_error.lower():
                wait = 20 * (attempt + 1)
                print(f'    Rate limited, waiting {wait}s...')
                time.sleep(wait)
            elif '402' in last_error:
                print(f'    ⚠️ Credits exhausted! Top up at https://openrouter.ai/settings/credits')
                raise
            elif attempt < 2:
                time.sleep(3)
    
    print(f'    Failed after 3 attempts: {last_error[:100] if last_error else "unknown"}')
    return []


def match_and_validate(ocr_rows, expected_rows, doc_type):
    result = {}
    ocr_lookup = {}
    for r in ocr_rows:
        name = r.get('party_name', '')
        votes = clean_votes(r.get('votes', '0'))
        if name:
            ocr_lookup[name] = votes
    
    used = set()
    for exp in expected_rows:
        exp_name = exp['party_name']
        sub_id = exp['id']
        
        if exp_name in ocr_lookup and exp_name not in used:
            result[sub_id] = validate_votes(ocr_lookup[exp_name])
            used.add(exp_name)
            continue
        
        best_name, best_score = None, 0
        for ocr_name in ocr_lookup:
            if ocr_name in used:
                continue
            score = max(
                fuzz.ratio(exp_name, ocr_name),
                fuzz.partial_ratio(exp_name, ocr_name),
                fuzz.token_sort_ratio(exp_name, ocr_name),
            )
            if score > best_score and score >= 60:
                best_score = score
                best_name = ocr_name
        
        if best_name:
            result[sub_id] = validate_votes(ocr_lookup[best_name])
            used.add(best_name)
        else:
            result[sub_id] = '0'
    
    return result


def validate_votes(votes_str):
    if not votes_str or votes_str == '0':
        return '0'
    try:
        n = int(votes_str)
        if n < 0: return '0'
    except ValueError:
        return '0'
    return votes_str


def load_checkpoint():
    if CHECKPOINT_FILE.exists():
        with open(CHECKPOINT_FILE, encoding='utf-8') as f:
            return json.load(f)
    return {}

def save_checkpoint(results):
    with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

def save_submission_csv(results):
    all_votes = {}
    for doc_id, expected_rows in doc_expected.items():
        ocr_rows = results.get(doc_id, [])
        if not ocr_rows:
            for exp in expected_rows:
                all_votes[exp['id']] = '0'
        else:
            doc_type = doc_types.get(doc_id, 'constituency')
            matched = match_and_validate(ocr_rows, expected_rows, doc_type)
            all_votes.update(matched)
    sub = template_df[['id']].copy()
    sub['votes'] = sub['id'].map(all_votes).fillna('0')
    csv_path = OUTPUT_DIR / 'submission.csv'
    sub.to_csv(csv_path, index=False)
    return csv_path

print(f'Engine ready ({MODEL} via OpenRouter).')

In [ ]:
# === Cell 7: TEST on 5 Sample Labels FIRST ===
# ทดสอบกับ 5 documents ที่มี ground truth ก่อน
# ลบ checkpoint เก่าเพื่อทดสอบ prompt ใหม่
import os as _os
if CHECKPOINT_FILE.exists():
    _os.remove(CHECKPOINT_FILE)
    print('Cleared old checkpoint for fresh test.')

def levenshtein(s1, s2):
    if len(s1) < len(s2): return levenshtein(s2, s1)
    if len(s2) == 0: return len(s1)
    prev = range(len(s2) + 1)
    for i, c1 in enumerate(s1):
        curr = [i + 1]
        for j, c2 in enumerate(s2):
            curr.append(min(prev[j+1]+1, curr[j]+1, prev[j]+(c1!=c2)))
        prev = curr
    return prev[-1]

sample_results = {}
total_dist = 0
total_pairs = 0
total_correct = 0

if SAMPLE_LABELS_DIR.exists():
    label_files = sorted(SAMPLE_LABELS_DIR.glob('*.json'))
    print(f'Testing on {len(label_files)} sample documents...\n')
    
    for jf in label_files:
        doc_id = jf.stem
        with open(jf, encoding='utf-8') as f:
            label = json.load(f)
        
        gt = {r.get('party', r.get('party_name', '')): str(r['votes']) for r in label['results']}
        doc_type = doc_types.get(doc_id, 'constituency')
        expected_parties = [e['party_name'] for e in doc_expected[doc_id]]
        
        print(f'--- {doc_id} ({doc_type}) ---')
        print(f'Pages: {len(documents[doc_id])}, Parties: {len(expected_parties)}')
        
        t0 = time.time()
        try:
            ocr_rows = extract_document(doc_id, documents[doc_id], doc_type, expected_parties)
            sample_results[doc_id] = ocr_rows
            elapsed = time.time() - t0
            print(f'OCR: {len(ocr_rows)} rows in {elapsed:.1f}s')
            
            # Match and validate
            matched = match_and_validate(ocr_rows, doc_expected[doc_id], doc_type)
            
            doc_dist = 0
            doc_correct = 0
            for exp in doc_expected[doc_id]:
                pred = matched.get(exp['id'], '0')
                actual = gt.get(exp['party_name'], '0')
                dist = levenshtein(str(pred), str(actual))
                doc_dist += dist
                total_dist += dist
                total_pairs += 1
                if dist == 0:
                    doc_correct += 1
                    total_correct += 1
                ok = '✓' if dist == 0 else '✗'
                if dist > 0:
                    print(f"  {ok} {exp['party_name']:<30s} actual={actual:>8s} pred={pred:>8s} dist={dist}")
            
            print(f'  Score: {doc_correct}/{len(expected_parties)} correct, mean dist={doc_dist/len(expected_parties):.3f}')
        
        except Exception as e:
            print(f'  ERROR: {e}')
        
        time.sleep(4)  # Rate limit
        print()
    
    if total_pairs > 0:
        print(f'\n{"="*60}')
        print(f'OVERALL: {total_correct}/{total_pairs} correct ({100*total_correct/total_pairs:.1f}%)')
        print(f'Mean Levenshtein: {total_dist/total_pairs:.4f}')
        print(f'{"="*60}')
else:
    print('No sample labels found!')

In [ ]:
# === Cell 8: Run ALL Documents ===
results = load_checkpoint()

# Add sample results to checkpoint
for doc_id, rows in sample_results.items():
    if doc_id not in results or not results[doc_id]:
        results[doc_id] = rows

good_results = {k: v for k, v in results.items() if v}
print(f'Loaded {len(good_results)} good results from checkpoint.')

doc_ids_sorted = sorted(documents.keys(), key=lambda d: (doc_types.get(d, 'z'), d))
todo = [d for d in doc_ids_sorted if d not in good_results]
print(f'To process: {len(todo)} (skipped {len(good_results)} done)')
print(f'Estimated: ~{len(todo)*4//60} minutes (15 RPM, 4s delay)\n')

SAVE_EVERY = 5
DELAY = 4  # 15 RPM = 4s between requests
errors = []
start_time = time.time()

for i, doc_id in enumerate(tqdm(todo, desc='OCR')):
    doc_type = doc_types.get(doc_id, 'constituency')
    page_paths = documents[doc_id]
    expected_parties = [e['party_name'] for e in doc_expected[doc_id]]

    try:
        rows = extract_document(doc_id, page_paths, doc_type, expected_parties)
        results[doc_id] = rows

        expected_count = template_doc_rows.get(doc_id, -1)
        if expected_count > 0 and len(rows) != expected_count:
            print(f'  WARN {doc_id}: got {len(rows)}, expected {expected_count}')

    except Exception as e:
        print(f'  Error {doc_id}: {str(e)[:80]}')
        results[doc_id] = []
        errors.append(doc_id)

    done = len([v for v in results.values() if v])
    if (i + 1) % SAVE_EVERY == 0 or i == len(todo) - 1:
        save_checkpoint(results)
        save_submission_csv(results)
        elapsed = time.time() - start_time
        remaining = (elapsed / max(i+1, 1)) * (len(todo) - i - 1)
        print(f'  [{done}/{len(documents)}] {elapsed/60:.1f}min, ~{remaining/60:.0f}min left')

    time.sleep(DELAY)

save_checkpoint(results)
csv_path = save_submission_csv(results)
elapsed = time.time() - start_time
done_total = len([v for v in results.values() if v])
print(f'\nDone! {done_total}/{len(documents)} docs in {elapsed/60:.1f} min')
if errors:
    print(f'Failed ({len(errors)}): {errors}')
print(f'Submission: {csv_path}')

In [ ]:
# === Cell 9: Retry Failed + Low-Quality Documents ===
results = load_checkpoint()

# Find docs that need retry
retry_docs = []
for doc_id in documents:
    if doc_id not in results:
        retry_docs.append(doc_id)
        continue
    ocr_rows = results[doc_id]
    expected = template_doc_rows.get(doc_id, -1)
    if expected > 0:
        # Flag if too many missing
        matched = match_and_validate(ocr_rows, doc_expected[doc_id], doc_types.get(doc_id, 'c'))
        zero_count = sum(1 for v in matched.values() if v == '0')
        if zero_count > expected * 0.3:  # >30% zeros = suspicious
            retry_docs.append(doc_id)

print(f'Documents to retry: {len(retry_docs)}')

if retry_docs:
    for doc_id in tqdm(retry_docs, desc='Retry'):
        doc_type = doc_types.get(doc_id, 'constituency')
        expected_parties = [e['party_name'] for e in doc_expected[doc_id]]
        try:
            rows = extract_document(doc_id, documents[doc_id], doc_type, expected_parties)
            results[doc_id] = rows
        except Exception as e:
            print(f'  Retry failed {doc_id}: {e}')
        time.sleep(4)
    
    save_checkpoint(results)
    save_submission_csv(results)
    print('Retry complete, CSV updated.')

In [ ]:
# === Cell 10: Final Validation & Download ===
results = load_checkpoint()

# Validate against sample labels
if SAMPLE_LABELS_DIR.exists():
    total_dist = 0
    total_pairs = 0
    for jf in sorted(SAMPLE_LABELS_DIR.glob('*.json')):
        doc_id = jf.stem
        with open(jf, encoding='utf-8') as f:
            label = json.load(f)
        gt = {r.get('party', r.get('party_name', '')): str(r['votes']) for r in label['results']}
        ocr_rows = results.get(doc_id, [])
        matched = match_and_validate(ocr_rows, doc_expected.get(doc_id, []), doc_types.get(doc_id, 'c'))
        for exp in doc_expected.get(doc_id, []):
            pred = matched.get(exp['id'], '0')
            actual = gt.get(exp['party_name'], '0')
            total_dist += levenshtein(str(pred), str(actual))
            total_pairs += 1
    print(f'Sample validation: mean Levenshtein = {total_dist/max(total_pairs,1):.4f} ({total_pairs} pairs)')

# Stats
sub = pd.read_csv(OUTPUT_DIR / 'submission.csv')
zeros = (sub['votes'] == '0').sum()
print(f'Submission: {len(sub)} rows, {zeros} zeros ({100*zeros/len(sub):.1f}%)')

# Download
from google.colab import files
files.download(str(OUTPUT_DIR / 'submission.csv'))
print('Downloaded!')